# Agentic RAG Platform — Local Demo Notebook

This notebook walks through a local, open-source version of the Agentic RAG platform.
It uses local substitutes for storage, search, and agents so you can run the core workflow with an OpenAI API key only.

## Notebook structure

1. **Pre-requisites & Environment Setup** — install Python packages and configure the `.env` file.
2. **Settings & Configuration** — load local settings and paths.
3. **Local Client Initialisation** — initialise LLM, Redis substitute, SQLite, and file storage.
4. **Schemas / Data Models** — define request and response shapes.
5. **Knowledge Base — Ingestion & Search** — build a ChromaDB vector store from local docs.
6. **Agent Tools** — implement local replacements for search, SQL queries, Redis history, and sandbox execution.
7. **SRE Agent** — run the reliability incident assistant.
8. **Engineering Agent** — run the architecture/code review assistant.
9. **Streamlit Chat UI** — launch a local browser interface.

---

## Pre-requisites

### 1. Python 3.10+

### 2. Install dependencies
```bash
pip install openai langchain langchain-openai langchain-text-splitters chromadb pydantic pydantic-settings httpx streamlit
```

### 3. OpenAI API key
Create a `.env` file in the project root with:
```dotenv
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o
CHROMA_EMBEDDING_FUNCTION=default
LLM_TEMPERATURE=0.2
```

> This notebook uses standard OpenAI and ChromaDB only; no Azure services are required.

## 0. Imports & Path Setup

In [4]:
import sys, os, json, logging, hashlib, asyncio, traceback, re
from pathlib import Path
from typing import Any, Optional
from enum import Enum

try:
    import tiktoken
except ImportError:
    tiktoken = None

In [5]:
# Ensure the project root is on sys.path so we can reference app/ structure
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", os.getcwd())
print("Project root    :", PROJECT_ROOT)

Working directory: c:\ML\AgenticAI\rag-infra\rag_infra
Project root    : c:\ML\AgenticAI\rag-infra\rag_infra


## 1. Settings & Configuration

This section loads local application settings from `.env` and defines paths for data, docs, and vector storage.


In [6]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """Local configuration for the notebook demo."""

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        env_ignore_empty=True,
        extra="ignore",
    )

    # sensitive info - set in .env or environment variables
    llm_provider: str = "openai"
    openai_api_key: str = ""
    openai_model: str = "gpt-4o"
    googlegemini_api_key: str = ""
    googlegemini_model: str = "gemini-1.5-flash"
    llm_temperature: float = 0.2

    # Local data paths
    data_dir: Path = Path("data")
    db_path: Path = Path("data/demo_local.db")
    docs_dir: Path = Path("data/raw-docs")
    chroma_rag_dir: Path = Path("data/chroma_rag")

    # RAG tuning
    rag_chunk_size: int = 800
    rag_chunk_overlap: int = 120
    rag_top_k: int = 5

    # Local ChromaDB embedding function selector.
    chroma_embedding_function: str = "default"


settings = Settings()

# Ensure directories exist
for d in [settings.data_dir, settings.docs_dir, settings.chroma_rag_dir]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

print(f"LLM provider     : {settings.llm_provider}")
print(f"OpenAI API key   : {'***' + settings.openai_api_key[-4:] if settings.openai_api_key else '(not set)'}")
print(f"OpenAI model     : {settings.openai_model}")
print(f"Google Gemini API key : {'***' + settings.googlegemini_api_key[-4:] if settings.googlegemini_api_key else '(not set)'}")
print(f"Google Gemini model   : {settings.googlegemini_model}")
print(f"ChromaDB embedding function: {settings.chroma_embedding_function}")
print(f"Docs directory   : {settings.docs_dir}")
print(f"ChromaDB path    : {settings.chroma_rag_dir}")
print(f"RAG chunk size   : {settings.rag_chunk_size}")
print(f"RAG top_k        : {settings.rag_top_k}")

LLM provider     : openai
OpenAI API key   : ***MJAA
OpenAI model     : gpt-4o
Google Gemini API key : (not set)
Google Gemini model   : gemini-1.5-flash
ChromaDB embedding function: default
Docs directory   : data\raw-docs
ChromaDB path    : data\chroma_rag
RAG chunk size   : 800
RAG top_k        : 5


## 2. Schemas / Data Models

These are the Pydantic models from `app/models/schemas.py`.  
They define the request/response shapes used by the agents and the UI.

In [7]:
from pydantic import BaseModel, Field


class AgentType(str, Enum):
    sre = "sre"
    engineering = "engineering"


class ChatRequest(BaseModel):
    agent: AgentType = AgentType.sre
    session_id: str = Field(..., description="Unique session/conversation ID")
    message: str = Field(..., min_length=1, max_length=4096)


class ChatResponse(BaseModel):
    session_id: str
    agent: AgentType
    answer: str
    sources: list[str] = []
    tool_calls: list[str] = []

class IngestRequest(BaseModel):
    container: str = "raw-docs"
    blob_prefix: str = ""
    force_reindex: bool = False


class IngestResponse(BaseModel):
    status: str
    documents_indexed: int
    errors: list[str] = []


class DocumentItem(BaseModel):
    name: str
    size: int
    last_modified: str
    uri: str


class DocumentsResponse(BaseModel):
    container: str
    documents: list[DocumentItem]


class ServiceStatus(str, Enum):
    ok = "ok"
    degraded = "degraded"
    error = "error"


class HealthResponse(BaseModel):
    status: ServiceStatus
    services: dict[str, Any]


# Quick validation
sample = ChatRequest(agent="sre", session_id="demo-001", message="Why is the API latency high?")
print("Sample ChatRequest:", sample.model_dump_json(indent=2))

Sample ChatRequest: {
  "agent": "sre",
  "session_id": "demo-001",
  "message": "Why is the API latency high?"
}


## 3. Local Client Initialisation

This section creates local infrastructure replacements for the deployed application.
It instantiates:
- a LangChain `ChatOpenAI` client for LLM calls
- an in-memory Redis substitute for conversation history
- a local SQLite database for incidents and dependencies
- a local document directory for ingestion and retrieval

In [8]:
import sqlite3
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

logger = logging.getLogger("demo")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


# ── 3a. LangChain ChatOpenAI LLM (standard OpenAI / Google Gemini) ───────
if settings.llm_provider.lower() == "gemini":
    if not settings.googlegemini_api_key:
        raise ValueError(
            "GOOGLE_GEMINI_API_KEY is not set. "
            "Please add it to your .env file in the project root."
        )
    llm_model = settings.googlegemini_model
    llm_api_key = settings.googlegemini_api_key
    llm = ChatGoogleGenerativeAI(
        model=llm_model,
        api_key=llm_api_key
    )
    print(f"✅ LangChain Google Gemini initialised (provider={settings.llm_provider} model={llm_model})")    
else:
    if not settings.openai_api_key:
        raise ValueError(
            "OPENAI_API_KEY is not set. "
            "Please add it to your .env file in the project root."
        )
    llm_model = settings.openai_model
    llm_api_key = settings.openai_api_key

    llm = ChatOpenAI(
        model=llm_model,
        api_key=llm_api_key,
        temperature=settings.llm_temperature,
    )
    print(f"✅ LangChain ChatOpenAI initialised (provider={settings.llm_provider} model={llm_model})")


# ── 3b. Local Redis substitute (in-memory dict) ─────────────────────────────
class LocalRedis:
    """Minimal Redis emulator for conversation history in this notebook."""

    def __init__(self):
        # Store values in memory; this is reset each notebook session.
        self._store: dict[str, str] = {}

    def get(self, key: str) -> Optional[str]:
        # Return stored JSON history or None when missing.
        return self._store.get(key)

    def setex(self, key: str, ttl: int, value: str):
        # Persist the value locally; TTL is ignored in this demo.
        self._store[key] = value

    def ping(self) -> bool:
        # Always healthy for the local demo.
        return True


redis_client = LocalRedis()
print("✅ Local Redis (in-memory dict) ready")


# ── 3c. Local SQLite substitute for PostgreSQL ──────────────────────────────
DB_PATH = PROJECT_ROOT / settings.db_path


def _get_db() -> sqlite3.Connection:
    """Open a connection to the local SQLite database."""
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def _init_local_pg():
    """Create local SQLite tables used by the notebook agents and tools."""
    with _get_db() as conn:
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS incidents (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                service TEXT NOT NULL,
                severity TEXT NOT NULL DEFAULT 'medium',
                title TEXT NOT NULL,
                started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                resolved_at TIMESTAMP,
                summary TEXT
            );
            CREATE TABLE IF NOT EXISTS service_dependencies (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                upstream TEXT NOT NULL,
                downstream TEXT NOT NULL,
                dependency_type TEXT NOT NULL DEFAULT 'http'
            );
            CREATE TABLE IF NOT EXISTS agent_interactions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                agent TEXT NOT NULL,
                query TEXT NOT NULL,
                answer TEXT NOT NULL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """)


_init_local_pg()
print(f"✅ Local SQLite DB initialised at {DB_PATH}")


# ── 3d. Local document directory ───────────────
LOCAL_DOCS_DIR = PROJECT_ROOT / settings.docs_dir
LOCAL_DOCS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Local docs directory: {LOCAL_DOCS_DIR}")
print("   Place .md / .txt files here for ingestion")

c:\Users\sayan\anaconda3\envs\py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LangChain ChatOpenAI initialised (provider=openai model=gpt-4o)
✅ Local Redis (in-memory dict) ready
✅ Local SQLite DB initialised at c:\ML\AgenticAI\rag-infra\rag_infra\data\demo_local.db
✅ Local docs directory: c:\ML\AgenticAI\rag-infra\rag_infra\data\raw-docs
   Place .md / .txt files here for ingestion


In [9]:
from langchain_core.messages import HumanMessage

print("## 3a-1. LLM connectivity test")
try:
    test_response = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
    print("LLM test successful")
    print("Response:", test_response.content)
except Exception as exc:
    print("LLM test failed:", exc)


## 3a-1. LLM connectivity test


INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LLM test successful
Response: Hello! How are you today?


### 3e. Seed sample data

Insert a few sample incidents and service dependencies so the agent tools have something to query.

In [10]:
with _get_db() as conn:
    # Seed incidents (idempotent — skip if rows exist)
    existing = conn.execute("SELECT COUNT(*) FROM incidents").fetchone()[0]
    if existing == 0:
        conn.executemany(
            "INSERT INTO incidents (service, severity, title, summary) VALUES (?, ?, ?, ?)",
            [
                ("payment-service", "high", "Payment gateway 5xx spike",
                 "Intermittent 502s from Stripe webhook handler; root cause was connection pool exhaustion."),
                ("auth-service", "critical", "OAuth token endpoint down",
                 "Expired TLS cert on auth-service caused 100% failures for 12 minutes."),
                ("order-service", "medium", "Slow order confirmation emails",
                 "SQS queue lag reached 45 s due to under-provisioned consumers."),
                ("payment-service", "low", "Minor logging noise in payment-service",
                 "Debug-level logs flooding CloudWatch; log level bumped to INFO."),
            ],
        )
        print("Seeded 4 sample incidents")
    else:
        print(f"Incidents table already has {existing} rows — skipping seed")

    # Seed service dependencies
    existing_deps = conn.execute("SELECT COUNT(*) FROM service_dependencies").fetchone()[0]
    if existing_deps == 0:
        conn.executemany(
            "INSERT INTO service_dependencies (upstream, downstream, dependency_type) VALUES (?, ?, ?)",
            [
                ("api-gateway", "auth-service", "http"),
                ("api-gateway", "order-service", "http"),
                ("order-service", "payment-service", "http"),
                ("order-service", "inventory-service", "grpc"),
                ("payment-service", "stripe-webhook", "http"),
                ("notification-service", "order-service", "async/sqs"),
            ],
        )
        print("Seeded 6 sample service dependencies")
    else:
        print(f"Dependencies table already has {existing_deps} rows — skipping seed")

Incidents table already has 4 rows — skipping seed
Dependencies table already has 6 rows — skipping seed


## 4. Agent Tools

This section implements local tool modules for the notebook agents:

- `search_tool` — hybrid retrieval using ChromaDB and local keyword search
- `postgres_tool` — incident history, service dependencies, and audit logs in SQLite
- `redis_tool` — conversation history and caching in an in-memory dict
- `sandbox_tool` — restricted Python execution for safe code evaluation

### 4a. Knowledge Base Service (ChromaDB)

This section builds a local ChromaDB-backed vector store from markdown and text files.
The notebook uses ChromaDB embeddings and chunked documents to support local retrieval.

In [11]:
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter


class KnowledgeBaseService:
    """Local ChromaDB knowledge base for semantic retrieval.

    Documents are chunked, embedded, and stored in a persistent ChromaDB collection.
    This local implementation supports the notebook's hybrid retrieval workflow.
    """

    def __init__(self, settings: Settings):
        self._settings = settings
        chroma_path = PROJECT_ROOT / settings.chroma_rag_dir
        chroma_path.mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(path=str(chroma_path))
        self._collection_name = "rag_knowledge_base"

        if settings.chroma_embedding_function == "default":
            self._embedding_function = embedding_functions.DefaultEmbeddingFunction()
        elif settings.chroma_embedding_function.startswith("all-"):
            self._embedding_function = embedding_functions.SBERTEmbeddingFunction(settings.chroma_embedding_function)
        else:
            self._embedding_function = embedding_functions.DefaultEmbeddingFunction()

        self._collection = self._client.get_or_create_collection(
            name=self._collection_name,
            embedding_function=self._embedding_function,
        )
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.rag_chunk_size,
            chunk_overlap=settings.rag_chunk_overlap,
        )

    def ingest_directory(self, directory: Path, clear_existing: bool = False) -> dict[str, int]:
        """Chunk and upsert .md / .txt files into ChromaDB."""
        if clear_existing:
            self._client.delete_collection(name=self._collection_name)
            self._collection = self._client.get_or_create_collection(
                name=self._collection_name,
                embedding_function=self._embedding_function,
            )

        source_files = sorted([*directory.glob("*.md"), *directory.glob("*.txt")])
        docs, ids, metadatas = [], [], []

        for file_path in source_files:
            try:
                text = file_path.read_text(encoding="utf-8", errors="ignore")
            except Exception as exc:
                logger.warning("Skipping %s due to read error: %s", file_path, exc)
                continue

            if not text.strip():
                logger.info("Skipping empty document: %s", file_path)
                continue

            chunks = self._splitter.split_text(text)
            for index, chunk in enumerate(chunks):
                chunk_hash = hashlib.sha1(chunk.encode("utf-8")).hexdigest()[:10]
                doc_id = f"{file_path.stem}-{index}-{chunk_hash}"
                docs.append(chunk)
                ids.append(doc_id)
                metadatas.append({"source": file_path.name, "chunk_index": index})

        if docs:
            try:
                self._collection.upsert(documents=docs, ids=ids, metadatas=metadatas)
            except Exception as exc:
                logger.warning("ChromaDB ingest failed: %s", exc)

        try:
            collection_count = self._collection.count()
        except Exception as exc:
            logger.warning("Failed to count ChromaDB collection: %s", exc)
            collection_count = -1

        return {
            "files_indexed": len(source_files),
            "chunks_indexed": len(docs),
            "collection_count": collection_count,
        }

    def search(self, query: str, top_k: int | None = None) -> list[dict[str, Any]]:
        """Query ChromaDB for relevant document chunks."""
        query = query.strip()
        if not query:
            return []

        try:
            collection_count = self._collection.count()
            if collection_count <= 0:
                return []

            results = self._collection.query(
                query_texts=[query],
                n_results=top_k or self._settings.rag_top_k,
                include=["documents", "metadatas", "distances"],
            )
        except Exception as exc:
            logger.warning("ChromaDB query failed: %s", exc)
            return []

        documents = (results.get("documents") or [[]])[0]
        metadatas = (results.get("metadatas") or [[]])[0]
        distances = (results.get("distances") or [[]])[0]
        combined = []
        for i, document in enumerate(documents):
            metadata = metadatas[i] if i < len(metadatas) else {}
            distance = distances[i] if i < len(distances) else None
            combined.append({
                "content": document,
                "source": metadata.get("source", "unknown"),
                "distance": distance,
            })
        return combined


    @property
    def count(self) -> int:
        try:
            return self._collection.count()
        except Exception as exc:
            logger.warning("Failed to retrieve ChromaDB count: %s", exc)
            return -1


# Initialise the knowledge base
kb = KnowledgeBaseService(settings=settings)
print(f"✅ ChromaDB knowledge base initialised at {PROJECT_ROOT / settings.chroma_rag_dir}")
print(f"   Collection: {kb._collection_name} ({kb.count} chunks)")

# Ingest any docs already in the local docs directory
ingest_stats = kb.ingest_directory(LOCAL_DOCS_DIR)
print(f"   Ingest result: {json.dumps(ingest_stats, indent=2)}")

✅ ChromaDB knowledge base initialised at c:\ML\AgenticAI\rag-infra\rag_infra\data\chroma_rag
   Collection: rag_knowledge_base (8 chunks)
   Ingest result: {
  "files_indexed": 4,
  "chunks_indexed": 8,
  "collection_count": 8
}


### 4a-1. Inspect Knowledge Base Data

Peek at the documents and chunks stored in ChromaDB to verify what the agents will search over.

In [12]:
# ── Inspect all data in the ChromaDB knowledge base ──────────────────────────

# 1. Collection overview
print(f"Collection name : {kb._collection_name}")
print(f"Total chunks    : {kb.count}")
print()

# 2. List all indexed sources and their chunk counts
all_data = kb._collection.get(include=["metadatas", "documents"])
source_counts: dict[str, int] = {}
for meta in all_data["metadatas"]:
    src = meta.get("source", "unknown")
    source_counts[src] = source_counts.get(src, 0) + 1

print("Indexed sources:")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src:40s}  {cnt:>4d} chunks")
print()

# 3. Show a sample of stored chunks (first 5)
sample_size = min(5, len(all_data["documents"]))
print(f"Sample chunks (showing {sample_size} of {len(all_data['documents'])}):")
print("-" * 80)
for i in range(sample_size):
    meta = all_data["metadatas"][i]
    doc = all_data["documents"][i]
    print(f"[{i}] source={meta.get('source', '?')}  chunk_index={meta.get('chunk_index', '?')}")
    print(f"    {doc[:200]}{'...' if len(doc) > 200 else ''}")
    print()

Collection name : rag_knowledge_base
Total chunks    : 8

Indexed sources:
  company-policy.md                            2 chunks
  product-faq.md                               3 chunks
  runbook_payment_service.md                   1 chunks
  technical-runbook.md                         2 chunks

Sample chunks (showing 5 of 8):
--------------------------------------------------------------------------------
[0] source=runbook_payment_service.md  chunk_index=0
    # Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Ver...

[1] source=company-policy.md  chunk_index=0
    # Acme Corp — Remote Work Policy

## Overview
This document outlines the remote work policy for all Acme Corp employees effective January 2026.

## Eligibility
- Full-time employees who have completed...

[2] source=company-policy.md  chunk_index=1
    ## Security
- VPN must b

### 4b. Postgres Tool (local SQLite version)

Mirrors `app/agents/tools/postgres_tool.py` — incident history, service dependencies, and audit logging.

In [13]:
def query_incident_history(service_name: str, limit: int = 10) -> list[dict]:
    """Query recent incidents for a given service (local SQLite version)."""
    try:
        with _get_db() as conn:
            rows = conn.execute(
                "SELECT id, service, severity, title, started_at, resolved_at, summary "
                "FROM incidents WHERE service = ? ORDER BY started_at DESC LIMIT ?",
                (service_name, limit),
            ).fetchall()
        return [dict(r) for r in rows]
    except Exception as exc:
        logger.warning("Incident history query failed for %s: %s", service_name, exc)
        return []


def query_service_dependencies(service_name: str) -> list[dict]:
    """Query service dependency graph (local SQLite version)."""
    try:
        with _get_db() as conn:
            rows = conn.execute(
                "SELECT upstream, downstream, dependency_type "
                "FROM service_dependencies WHERE upstream = ? OR downstream = ?",
                (service_name, service_name),
            ).fetchall()
        return [dict(r) for r in rows]
    except Exception as exc:
        logger.warning("Service dependency query failed for %s: %s", service_name, exc)
        return []


def log_agent_interaction(session_id: str, agent: str, query: str, answer: str):
    """Persist agent interaction for audit (local SQLite version)."""
    try:
        with _get_db() as conn:
            conn.execute(
                "INSERT INTO agent_interactions (session_id, agent, query, answer) VALUES (?, ?, ?, ?)",
                (session_id, agent, query, answer),
            )
    except Exception as exc:
        logger.warning("Failed to log agent interaction for %s: %s", session_id, exc)


# Test the tools
print("Incidents for 'payment-service':")
for inc in query_incident_history("payment-service"):
    print(f"  [{inc['severity']}] {inc['title']}")

print("\nDependencies involving 'order-service':")
for dep in query_service_dependencies("order-service"):
    print(f"  {dep['upstream']} → {dep['downstream']} ({dep['dependency_type']})")

Incidents for 'payment-service':
  [high] Payment gateway 5xx spike
  [low] Minor logging noise in payment-service

Dependencies involving 'order-service':
  api-gateway → order-service (http)
  order-service → payment-service (http)
  order-service → inventory-service (grpc)
  notification-service → order-service (async/sqs)


### 4c. Redis Tool (local in-memory version)

Mirrors `app/agents/tools/redis_tool.py` — conversation history and caching.

In [14]:
HISTORY_TTL = 3600  # 1 hour (ignored locally, kept for parity)


def get_conversation_history(session_id: str) -> list[dict]:
    """Retrieve conversation history from local Redis substitute."""
    try:
        raw = redis_client.get(f"session:{session_id}:history")
        if not raw:
            return []
        return json.loads(raw)
    except json.JSONDecodeError:
        logger.warning("Corrupt conversation history for %s, resetting history.", session_id)
        return []
    except Exception as exc:
        logger.warning("Failed to retrieve conversation history for %s: %s", session_id, exc)
        return []


def append_to_history(session_id: str, role: str, content: str):
    """Append a message to the conversation history (keeps last 20 turns)."""
    try:
        history = get_conversation_history(session_id)
        history.append({"role": role, "content": content})
        history = history[-20:]
        redis_client.setex(
            f"session:{session_id}:history",
            HISTORY_TTL,
            json.dumps(history),
        )
    except Exception as exc:
        logger.warning("Failed to append conversation history for %s: %s", session_id, exc)


def cache_set(key: str, value: str, ttl: int = 300):
    try:
        redis_client.setex(key, ttl, value)
    except Exception as exc:
        logger.warning("Failed to set cache key %s: %s", key, exc)


def cache_get(key: str) -> Optional[str]:
    try:
        return redis_client.get(key)
    except Exception as exc:
        logger.warning("Failed to get cache key %s: %s", key, exc)
        return None


# Test
append_to_history("demo-001", "user", "Hello, what is the payment-service?")
append_to_history("demo-001", "assistant", "The payment-service handles all payment processing.")
history = get_conversation_history("demo-001")
print(f"Conversation history ({len(history)} messages):")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:80]}")

Conversation history (2 messages):
  [user] Hello, what is the payment-service?
  [assistant] The payment-service handles all payment processing.


### 4d. Sandbox Tool

Executes untrusted Python code in a restricted sandbox.  
This is the same implementation as `app/agents/tools/sandbox_tool.py` — no Azure dependency.

In [15]:
import builtins as _builtins_mod

SANDBOX_TIMEOUT = 10  # seconds
MAX_SANDBOX_CODE_LENGTH = 2000
MAX_SANDBOX_OUTPUT_CHARS = 5000
ALLOWED_BUILTINS = {
    "print", "len", "range", "enumerate", "zip",
    "list", "dict", "set", "tuple", "str", "int",
    "float", "bool", "type", "isinstance", "min", "max", "sum",
}


def execute_code(code: str) -> dict[str, Any]:
    """
    Execute untrusted Python code in a restricted sandbox.
    No file I/O, no imports, no network — pure computation only.
    (Synchronous version for notebook use; production uses async.)
    """
    if len(code) > MAX_SANDBOX_CODE_LENGTH:
        return {
            "status": "error",
            "output": f"Sandbox code block is too large ({len(code)} chars).",
        }

    restricted_globals = {
        "__builtins__": {k: getattr(_builtins_mod, k) for k in ALLOWED_BUILTINS if hasattr(_builtins_mod, k)},
    }
    output_lines: list[str] = []

    def _capture_print(*args, **kwargs):
        output_lines.append(" ".join(str(a) for a in args))

    restricted_globals["__builtins__"]["print"] = _capture_print

    try:
        local_vars: dict = {}
        exec(compile(code, "<sandbox>", "exec"), restricted_globals, local_vars)
        output = "\n".join(output_lines)
        if len(output) > MAX_SANDBOX_OUTPUT_CHARS:
            output = output[:MAX_SANDBOX_OUTPUT_CHARS] + "\n...[truncated]"
        return {
            "status": "ok",
            "output": output,
            "locals": {k: repr(v) for k, v in local_vars.items()},
        }
    except Exception:
        return {"status": "error", "output": traceback.format_exc(limit=5)}


# Test
result = execute_code("x = sum(range(10))\nprint('Sum:', x)")
print("Sandbox result:", json.dumps(result, indent=2))

Sandbox result: {
  "status": "ok",
  "output": "Sum: 45",
  "locals": {
    "x": "45"
  }
}


## 5. RAG Service — Ingestion & Retrieval

This section implements local retrieval and ingestion.
The code here is responsible for finding the most relevant runbook and documentation snippets for a user query, building a grounded context, and passing that context into the agents.
`retrieve_context()` combines keyword search and ChromaDB semantic search to build a context string.
`ingest_local_docs()` reads `.md`/`.txt` files from `data/raw-docs/` and stores them in ChromaDB.

In [18]:
CONTEXT_MAX_CHARS = 6000
MAX_QUERY_LENGTH = 2000
MAX_USER_MESSAGE_CHARS = 3000
MAX_HISTORY_TURNS = 10
MAX_PROMPT_TOKENS = 6500
MAX_RESPONSE_TOKENS = 1024
LLM_COST_PER_1K = 0.0025  # Approximate cost in USD per 1k tokens
# Common English words to skip during service name detection
_STOP_WORDS = {
    "what", "when", "where", "which", "there", "their", "about",
    "would", "could", "should", "have", "been", "that", "this",
    "with", "from", "your", "into", "will", "more", "also",
}


def _get_tokenizer(model_name: str = "gpt-4o"):
    if tiktoken is None:
        return None
    try:
        return tiktoken.encoding_for_model(model_name)
    except Exception:
        try:
            return tiktoken.get_encoding("cl100k_base")
        except Exception:
            return None

TOKENIZER = _get_tokenizer(settings.openai_model or "gpt-4o")


def count_tokens(text: str, model_name: str = None) -> int:
    if not text:
        return 0
    tokenizer = TOKENIZER
    if tokenizer is not None:
        try:
            return len(tokenizer.encode(text))
        except Exception:
            pass
    return max(1, len(text) // 4)


def count_messages_tokens(messages: list[Any], model_name: str = None) -> int:
    total = 0
    for message in messages:
        content = getattr(message, "content", message)
        total += count_tokens(str(content), model_name)
    return total


def estimate_cost(tokens: int, price_per_1k: float = LLM_COST_PER_1K) -> float:
    return round(tokens * price_per_1k / 1000.0, 6)


def validate_answer_against_sources(answer: str, sources: list[str]) -> dict[str, Any]:
    issues: list[str] = []
    normalized_answer = answer.lower().strip()

    if not normalized_answer:
        issues.append("Answer is empty.")

    if sources:
        normalized_sources = [source.lower() for source in sources]
        if not any(source in normalized_answer for source in normalized_sources):
            issues.append("Answer does not explicitly reference any retrieved source.")
    else:
        if any(token in normalized_answer for token in ["according to", "as described", "in the document", "source"]):
            issues.append("Answer references sources even though none were retrieved.")
        if "i don't know" not in normalized_answer and "unable to" not in normalized_answer and not issues:
            issues.append("No retrieved sources were available to ground the answer.")

    status = "supported"
    if issues:
        status = "unverified" if not sources else "partially_supported"
    if not sources and not issues:
        status = "unknown"

    return {
        "status": status,
        "issues": issues,
    }


def sanitize_user_message(message: str) -> str:
    """Sanitize user input before using it for retrieval or model prompts."""
    if message is None:
        return ""
    cleaned = message.replace("\r", " ").replace("\t", " ").replace("\x0b", " ").replace("\x0c", " ")
    return " ".join(cleaned.split())[:MAX_QUERY_LENGTH]


def _tokenize_search_terms(text: str) -> list[str]:
    terms = [token.lower() for token in re.findall(r"\w+", text)]
    return [term for term in terms if len(term) > 2 and term not in _STOP_WORDS]


def _keyword_search_local_docs(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    query_terms = _tokenize_search_terms(query)
    if not query_terms:
        return []

    # Search local raw docs for query terms and rank matching chunks by frequency.
    candidates: list[dict[str, Any]] = []
    source_files = sorted(LOCAL_DOCS_DIR.glob("*.md")) + sorted(LOCAL_DOCS_DIR.glob("*.txt"))
    for file_path in source_files:
        try:
            text = file_path.read_text(encoding="utf-8", errors="ignore")
        except Exception as exc:
            logger.warning("Failed to read %s for keyword search: %s", file_path, exc)
            continue

        for chunk_index, chunk in enumerate(kb._splitter.split_text(text)):
            score = sum(chunk.lower().count(term) for term in query_terms)
            if score > 0:
                candidates.append({
                    "content": chunk,
                    "source": file_path.name,
                    "score": score,
                    "chunk_index": chunk_index,
                })

    candidates.sort(key=lambda item: (-item["score"], item["source"], item["chunk_index"]))
    return candidates[:top_k]


def retrieve_context(query: str, top_k: int = 5) -> tuple[str, list[str]]:
    """
    Retrieve relevant chunks from ChromaDB and build a context string.
    Uses local hybrid search: keyword matching over raw docs plus semantic similarity via ChromaDB.
    """
    query = sanitize_user_message(query)
    if not query:
        return "", []

    # 1) Get exact keyword matches from raw local files.
    keyword_hits = _keyword_search_local_docs(query, top_k=top_k)
    try:
        # 2) Get semantic matches from the vector store.
        semantic_hits = kb.search(query, top_k=top_k)
    except Exception as exc:
        logger.warning("RAG context retrieval failed: %s", exc)
        semantic_hits = []

    combined: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()
    for hit in keyword_hits + semantic_hits:
        key = (hit["source"], hit["content"])
        if key in seen:
            continue
        seen.add(key)
        combined.append(hit)
        if len(combined) >= top_k:
            break

    context_parts = []
    sources = []
    total_chars = 0

    for hit in combined:
        chunk = f"[{hit['source']}]\n{hit['content']}"
        if total_chars + len(chunk) > CONTEXT_MAX_CHARS:
            break
        context_parts.append(chunk)
        sources.append(hit["source"])
        total_chars += len(chunk)

    context = "\n\n---\n\n".join(context_parts)
    return context, list(dict.fromkeys(sources))


def ingest_local_docs(directory: Path | None = None, clear_existing: bool = False) -> dict[str, int]:
    """Ingest local `.md` and `.txt` files into ChromaDB."""
    target = directory or LOCAL_DOCS_DIR
    try:
        return kb.ingest_directory(target, clear_existing=clear_existing)
    except Exception as exc:
        logger.warning("Local ingestion failed: %s", exc)
        return {"files_indexed": 0, "chunks_indexed": 0, "collection_count": kb.count}


# Create a sample document for testing if the docs directory is empty
sample_doc = LOCAL_DOCS_DIR / "runbook_payment_service.md"
if not sample_doc.exists():
    sample_doc.write_text(
        "# Payment Service Runbook\n\n"
        "## Overview\n"
        "The payment-service processes all payment transactions via Stripe.\n\n"
        "## Common Issues\n"
        "### 5xx Errors\n"
        "- Check connection pool settings in `config/pool.yaml`.\n"
        "- Verify Stripe API key is valid and not rate-limited.\n"
        "- Inspect CloudWatch logs for `PoolExhausted` exceptions.\n\n"
        "### High Latency\n"
        "- Check database connection pool utilisation.\n"
        "- Review recent deployments for regression.\n"
        "- Verify downstream Stripe endpoint health at https://status.stripe.com.\n\n"
        "## Escalation\n"
        "If unresolved within 15 minutes, page the payments-oncall rotation.\n",
        encoding="utf-8",
    )
    print("Created sample runbook:", sample_doc.name)
    ingest_result = ingest_local_docs()
    print("Re-ingest result:", json.dumps(ingest_result, indent=2))

# Test retrieval
context, sources = retrieve_context("payment service 5xx errors")
print(f"\nRetrieved context ({len(context)} chars) from {len(sources)} source(s)")
if context:
    print(context[:500])


Retrieved context (3397 chars) from 4 source(s)
[runbook_payment_service.md]
# Payment Service Runbook

## Overview
The payment-service processes all payment transactions via Stripe.

## Common Issues
### 5xx Errors
- Check connection pool settings in `config/pool.yaml`.
- Verify Stripe API key is valid and not rate-limited.
- Inspect CloudWatch logs for `PoolExhausted` exceptions.

### High Latency
- Check database connection pool utilisation.
- Review recent deployments for regression.
- Verify downstream Stripe endpoint health at https://s


## 6. SRE Agent

The SRE agent (`app/agents/sre_agent.py`) analyses incidents, suggests root cause analysis, and helps with on-call triage.

It combines:
1. Conversation history (local Redis substitute)
2. RAG context (local keyword search)
3. Incident history (local SQLite)
4. Sandbox execution (if code block in message)
5. LLM call via **LangChain ChatOpenAI** (standard OpenAI API)

In [19]:
from langchain_core.messages import HumanMessage, SystemMessage

SRE_SYSTEM_PROMPT = """
You are an expert SRE (Site Reliability Engineer) AI assistant.
Your responsibilities:
- Analyze incidents and alerts based on historical data and runbooks.
- Suggest root cause analysis (RCA) and remediation steps.
- Help with on-call triage, runbook lookup, and postmortem drafting.
- Answer questions about service dependencies and SLOs/SLIs.
- Execute diagnostic code snippets safely when needed.

Always:
- Ground your answers in retrieved context from the knowledge base.
- Cite sources when referencing runbooks or past incidents.
- Be concise, structured (use numbered steps for procedures).
- If the answer cannot be supported by the retrieved context, say so clearly.
- Never reveal secrets, connection strings, or internal credentials.
"""

# Common English words to skip during service name detection
_STOP_WORDS = {
    "what", "when", "where", "which", "there", "their", "about",
    "would", "could", "should", "have", "been", "that", "this",
    "with", "from", "your", "into", "will", "more", "also",
}


def run_sre_agent(session_id: str, user_message: str, react: bool = False) -> dict[str, Any]:
    """
    Local synchronous version of app/agents/sre_agent.py.
    Uses local tools instead of Azure services.
    Calls OpenAI via LangChain ChatOpenAI.
    """
    user_message = sanitize_user_message(user_message)[:MAX_USER_MESSAGE_CHARS]
    logger.info("SRE agent: session=%s query='%s'", session_id, user_message[:80])

    # 1) Conversation history from local Redis
    history = []
    try:
        history = get_conversation_history(f"sre:{session_id}")
    except Exception as exc:
        logger.warning("History fetch failed: %s", exc)

    # 2) RAG context from local search index
    context, sources = retrieve_context(user_message, top_k=settings.rag_top_k)

    # 3) Incident lookup from local SQLite
    incidents = []
    tool_calls = []
    words = user_message.lower().split()
    for word in words:
        if len(word) > 4 and word not in _STOP_WORDS:
            try:
                rows = query_incident_history(word)
            except Exception as exc:
                logger.warning("Incident lookup failed for %s: %s", word, exc)
                rows = []
            if rows:
                incidents = rows
                tool_calls.append(f"query_incident_history(service={word})")
                break

    # 4) Sandbox execution (if code block detected)
    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    # 5) Build LangChain messages
    messages = [SystemMessage(content=SRE_SYSTEM_PROMPT)]
    if react:
        messages.append(SystemMessage(content=REACTION_INSTRUCTIONS))

    if context:
        messages.append(SystemMessage(content=f"Relevant knowledge base context:\n\n{context}"))
    else:
        messages.append(SystemMessage(content="No relevant documentation was retrieved. Answer only if you can be certain, otherwise say that you cannot find a supported answer."))

    if incidents:
        incident_text = "\n".join(
            f"- [{r['severity']}] {r['title']} at {r['started_at']}: {r.get('summary', '')}"
            for r in incidents
        )
        messages.append(SystemMessage(content=f"Recent incidents:\n{incident_text}"))

    if sandbox_result:
        messages.append(SystemMessage(
            content=f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
        ))

    for msg in history[-MAX_HISTORY_TURNS:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            from langchain_core.messages import AIMessage
            messages.append(AIMessage(content=msg["content"]))

    messages.append(HumanMessage(content=user_message))

    prompt_tokens = count_messages_tokens(messages, settings.openai_model)
    if prompt_tokens > MAX_PROMPT_TOKENS:
        logger.warning("Prompt token budget exceeded (%s), trimming history.", prompt_tokens)
        messages = [SystemMessage(content=SRE_SYSTEM_PROMPT)]
        if react:
            messages.append(SystemMessage(content=REACTION_INSTRUCTIONS))
        if context:
            messages.append(SystemMessage(content=f"Relevant knowledge base context:\n\n{context}"))
        if incidents:
            messages.append(SystemMessage(content=f"Recent incidents:\n{incident_text}"))
        if sandbox_result:
            messages.append(SystemMessage(
                content=f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
            ))
        for msg in history[-5:]:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            else:
                from langchain_core.messages import AIMessage
                messages.append(AIMessage(content=msg["content"]))
        messages.append(HumanMessage(content=user_message))
        prompt_tokens = count_messages_tokens(messages, settings.openai_model)

    completion_tokens = 0
    total_tokens = prompt_tokens
    try:
        response = llm.invoke(messages)
        answer = response.content
        llm_output = getattr(response, "llm_output", {}) or {}
        usage = llm_output.get("usage", {}) or {}
        prompt_tokens = usage.get("prompt_tokens", prompt_tokens)
        completion_tokens = usage.get("completion_tokens", count_tokens(answer, settings.openai_model))
        total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)
    except Exception as exc:
        logger.warning("LLM call failed: %s", exc)
        answer = (
            f"[LLM call failed: {exc}]\n\n"
            f"Context retrieved:\n{context[:500] if context else 'None'}"
        )
        completion_tokens = 0
        total_tokens = prompt_tokens

    estimated_cost = estimate_cost(total_tokens)
    validation = validate_answer_against_sources(answer, sources)

    try:
        append_to_history(f"sre:{session_id}", "user", user_message)
        append_to_history(f"sre:{session_id}", "assistant", answer)
    except Exception as exc:
        logger.warning("History save failed: %s", exc)

    try:
        log_agent_interaction(session_id, "sre", user_message, answer)
    except Exception as exc:
        logger.warning("Interaction log failed: %s", exc)

    return {
        "answer": answer,
        "sources": sources,
        "tool_calls": tool_calls,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost": estimated_cost,
        "validation": validation,
    }


print("✅ SRE agent defined")


✅ SRE agent defined


### 6a. Test the SRE Agent

In [20]:
sre_result = run_sre_agent(
    session_id="demo-sre-001",
    user_message="We're seeing 5xx errors on payment-service. What should I check first?",
)

print("=" * 60)
print("SRE AGENT RESPONSE")
print("=" * 60)
print(sre_result["answer"])
print(f"\nSources: {sre_result['sources']}")
print(f"Tool calls: {sre_result['tool_calls']}")
print(f"Prompt tokens: {sre_result['prompt_tokens']}")
print(f"Completion tokens: {sre_result['completion_tokens']}")
print(f"Total tokens: {sre_result['total_tokens']}")
print(f"Estimated cost: ${sre_result['estimated_cost']}")

INFO | SRE agent: session=demo-sre-001 query='We're seeing 5xx errors on payment-service. What should I check first?'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


SRE AGENT RESPONSE
To address 5xx errors on the payment-service, follow these steps:

1. **Check Connection Pool Settings**: Review the configuration in `config/pool.yaml` to ensure the connection pool settings are correct.

2. **Verify Stripe API Key**: Ensure that the Stripe API key is valid and not rate-limited. This can often be a cause of 5xx errors if the key is invalid or has exceeded its usage limits.

3. **Inspect CloudWatch Logs**: Look for `PoolExhausted` exceptions in the CloudWatch logs. This can indicate issues with the connection pool being exhausted, leading to 5xx errors.

If these steps do not resolve the issue within 15 minutes, escalate by paging the payments-oncall rotation as per the escalation policy. 

(Source: [runbook_payment_service.md])

Sources: ['runbook_payment_service.md', 'technical-runbook.md', 'company-policy.md', 'product-faq.md']
Tool calls: []
Prompt tokens: 988
Completion tokens: 168
Total tokens: 1156
Estimated cost: $0.00289


### 6b. SRE Agent — Latency Spike Investigation

In [21]:
sre_result_2 = run_sre_agent(
    session_id="demo-sre-002",
    user_message="We're observing p99 latency spikes on order-service above 2s. How should I triage this?",
)

print("=" * 60)
print("SRE AGENT RESPONSE — Latency Spike")
print("=" * 60)
print(sre_result_2["answer"])
print(f"\nSources: {sre_result_2['sources']}")
print(f"Tool calls: {sre_result_2['tool_calls']}")
print(f"Prompt tokens: {sre_result_2['prompt_tokens']}")
print(f"Completion tokens: {sre_result_2['completion_tokens']}")
print(f"Total tokens: {sre_result_2['total_tokens']}")
print(f"Estimated cost: ${sre_result_2['estimated_cost']}")

INFO | SRE agent: session=demo-sre-002 query='We're observing p99 latency spikes on order-service above 2s. How should I triag'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


SRE AGENT RESPONSE — Latency Spike
To triage the p99 latency spikes on the order-service, follow these steps:

1. **Check Recent Deployments:**
   - Review any recent deployments to the order-service that might have introduced regressions. Look for changes in code or configuration that could affect performance.

2. **Database Connection Pool:**
   - Verify the database connection pool utilization. High utilization can lead to latency spikes if the pool is exhausted. Adjust the pool size if necessary.

3. **Downstream Service Health:**
   - Check the health of any downstream services that the order-service depends on. For example, if the payment-service is involved, verify the health of the Stripe endpoint at [Stripe Status](https://status.stripe.com).

4. **Inspect Logs:**
   - Examine the application logs for any errors or warnings that could indicate the root cause of the latency. Look for patterns or repeated errors.

5. **Resource Utilization:**
   - Monitor CPU, memory, and networ

### 6c. SRE Agent — Runbook Lookup with Code Execution

In [22]:
sre_result_3 = run_sre_agent(
    session_id="demo-sre-003",
    user_message=(
        "Our monitoring shows Redis connection pool exhaustion on notification-service. "
        "Can you pull up the relevant runbook steps and also run this diagnostic?\n\n"
        "```python\nstats = {'active_connections': 48, 'max_pool': 50, 'idle': 2}\n"
        "utilization = stats['active_connections'] / stats['max_pool'] * 100\n"
        "print(f'Pool utilization: {utilization:.1f}%')\n"
        "print('CRITICAL' if utilization > 90 else 'OK')\n```"
    ),
)

print("=" * 60)
print("SRE AGENT RESPONSE — Runbook + Sandbox Execution")
print("=" * 60)
print(sre_result_3["answer"])
print(f"\nSources: {sre_result_3['sources']}")
print(f"Tool calls: {sre_result_3['tool_calls']}")
print(f"Prompt tokens: {sre_result_3['prompt_tokens']}")
print(f"Completion tokens: {sre_result_3['completion_tokens']}")
print(f"Total tokens: {sre_result_3['total_tokens']}")
print(f"Estimated cost: ${sre_result_3['estimated_cost']}")

INFO | SRE agent: session=demo-sre-003 query='Our monitoring shows Redis connection pool exhaustion on notification-service. C'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


SRE AGENT RESPONSE — Runbook + Sandbox Execution
To address the Redis connection pool exhaustion issue on the notification-service, you can follow these general steps, although specific runbook details for Redis are not available in the retrieved context:

1. **Check Connection Pool Settings**: Ensure that the connection pool settings are correctly configured to handle the expected load. This might involve increasing the maximum number of connections if the current limit is too low.

2. **Inspect Logs**: Review the application logs for any errors or warnings related to Redis connections. This can provide insights into whether the issue is due to configuration, network problems, or something else.

3. **Monitor Redis Metrics**: Use Redis monitoring tools to check for metrics like connection count, memory usage, and command execution time. This can help identify if the exhaustion is due to high load or inefficient queries.

4. **Review Recent Changes**: If the issue started recently, con

## 7. Engineering Agent

The Engineering agent (`app/agents/engineering_agent.py`) answers architecture, design, and code-related questions.

It combines:
1. Conversation history (local Redis substitute)
2. RAG context (local keyword search)
3. Service dependency lookup (local SQLite)
4. Sandbox execution (if code block in message)
5. LLM call via **LangChain ChatOpenAI** (standard OpenAI API)

In [23]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

ENGINEERING_SYSTEM_PROMPT = """
You are an expert Software Engineering AI assistant embedded in an engineering platform.
Your responsibilities:
- Answer architecture, design, and code-related questions.
- Review code snippets and suggest improvements.
- Explain service dependencies and integration patterns.
- Help with debugging, performance analysis, and best practices.
- Execute safe code snippets in a sandboxed environment.

Always:
- Ground answers in retrieved internal documentation.
- Cite sources (ADRs, RFCs, wiki pages) when relevant.
- Be precise and use concrete examples.
- Prefer idiomatic, production-ready code suggestions.
- If the answer cannot be supported by the retrieved context, say so clearly.
- Never output secrets, credentials, or connection strings.
"""


def run_engineering_agent(session_id: str, user_message: str, react: bool = False) -> dict[str, Any]:
    """
    Local synchronous version of app/agents/engineering_agent.py.
    Uses local tools instead of Azure services.
    Calls OpenAI via LangChain ChatOpenAI.
    """
    user_message = sanitize_user_message(user_message)[:MAX_USER_MESSAGE_CHARS]
    logger.info("Engineering agent: session=%s query='%s'", session_id, user_message[:80])

    # 1) Conversation history
    history = []
    try:
        history = get_conversation_history(f"eng:{session_id}")
    except Exception as exc:
        logger.warning("History fetch failed: %s", exc)

    # 2) RAG context
    context, sources = retrieve_context(user_message, top_k=settings.rag_top_k)

    # 3) Service dependency lookup
    dependencies = []
    tool_calls = []
    words = user_message.lower().split()
    for word in words:
        if len(word) > 4 and word not in _STOP_WORDS:
            try:
                rows = query_service_dependencies(word)
            except Exception as exc:
                logger.warning("Dependency lookup failed for %s: %s", word, exc)
                rows = []
            if rows:
                dependencies = rows
                tool_calls.append(f"query_service_dependencies(service={word})")
                break

    # 4) Sandbox execution (if code block detected)
    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    # 5) Build LangChain messages
    messages = [SystemMessage(content=ENGINEERING_SYSTEM_PROMPT)]
    if react:
        messages.append(SystemMessage(content=REACTION_INSTRUCTIONS))

    if context:
        messages.append(SystemMessage(content=f"Relevant internal documentation:\n\n{context}"))
    else:
        messages.append(SystemMessage(content="No relevant documentation was retrieved. Answer only if you can be certain, otherwise say that you cannot find a supported answer."))

    if dependencies:
        dep_text = "\n".join(
            f"- {r['upstream']} → {r['downstream']} ({r['dependency_type']})"
            for r in dependencies
        )
        messages.append(SystemMessage(content=f"Service dependencies:\n{dep_text}"))

    if sandbox_result:
        messages.append(SystemMessage(
            content=f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
        ))

    for msg in history[-MAX_HISTORY_TURNS:]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))

    messages.append(HumanMessage(content=user_message))

    prompt_tokens = count_messages_tokens(messages, settings.openai_model)
    if prompt_tokens > MAX_PROMPT_TOKENS:
        logger.warning("Prompt token budget exceeded (%s), trimming history.", prompt_tokens)
        messages = [SystemMessage(content=ENGINEERING_SYSTEM_PROMPT)]
        if react:
            messages.append(SystemMessage(content=REACTION_INSTRUCTIONS))
        if context:
            messages.append(SystemMessage(content=f"Relevant internal documentation:\n\n{context}"))
        if dependencies:
            messages.append(SystemMessage(content=f"Service dependencies:\n{dep_text}"))
        if sandbox_result:
            messages.append(SystemMessage(
                content=f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}",
            ))
        for msg in history[-5:]:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            else:
                messages.append(AIMessage(content=msg["content"]))
        messages.append(HumanMessage(content=user_message))
        prompt_tokens = count_messages_tokens(messages, settings.openai_model)

    completion_tokens = 0
    total_tokens = prompt_tokens
    try:
        response = llm.invoke(messages)
        answer = response.content
        llm_output = getattr(response, "llm_output", {}) or {}
        usage = llm_output.get("usage", {}) or {}
        prompt_tokens = usage.get("prompt_tokens", prompt_tokens)
        completion_tokens = usage.get("completion_tokens", count_tokens(answer, settings.openai_model))
        total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)
    except Exception as exc:
        logger.warning("LLM call failed: %s", exc)
        answer = (
            f"[LLM call failed: {exc}]\n\n"
            f"Context retrieved:\n{context[:500] if context else 'None'}"
        )
        completion_tokens = 0
        total_tokens = prompt_tokens

    estimated_cost = estimate_cost(total_tokens)
    validation = validate_answer_against_sources(answer, sources)

    try:
        append_to_history(f"eng:{session_id}", "user", user_message)
        append_to_history(f"eng:{session_id}", "assistant", answer)
    except Exception as exc:
        logger.warning("History save failed: %s", exc)

    try:
        log_agent_interaction(session_id, "engineering", user_message, answer)
    except Exception as exc:
        logger.warning("Interaction log failed: %s", exc)

    return {
        "answer": answer,
        "sources": sources,
        "tool_calls": tool_calls,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost": estimated_cost,
        "validation": validation,
    }


print("✅ Engineering agent defined")


✅ Engineering agent defined


### 7. RAGAS: Topic-Based Retrieval as a Service

In [24]:
RAGAS_TOPIC_SYSTEM_PROMPT = """
You are a topic-focused Retrieval-Augmented Generation service. Use the provided topic and retrieved documentation to answer the user clearly and accurately.
Always ground your answer in the retrieved context, cite sources when relevant, and avoid making up details that are not supported.
"""

RAGAS_TOPIC_FILTERED_SYSTEM_PROMPT = """
You are a topic-filtered RAG orchestration assistant. Use the provided topic, retrieved documents, and any relevant tool outputs to answer the user.
If the topic appears to be an internal service, enrich the response with service dependency information. If a Python code block appears, execute it safely in the sandbox and include the result.
Always ground answers in retrieved sources and identify when tool results were used.
"""


def run_ragas_topic_service(topic: str, user_message: str) -> dict[str, Any]:
    user_message = sanitize_user_message(user_message)[:MAX_USER_MESSAGE_CHARS]
    logger.info("RAGAS topic service: topic=%s query='%s'", topic, user_message[:80])

    context, sources = retrieve_context(f"{topic} {user_message}", top_k=settings.rag_top_k)
    messages = [SystemMessage(content=RAGAS_TOPIC_SYSTEM_PROMPT)]

    if context:
        messages.append(SystemMessage(content=f"Topic: {topic}\nRelevant documentation:\n{context}"))
    else:
        messages.append(SystemMessage(content=(
            f"Topic: {topic}\nNo relevant documentation was retrieved. "
            "Answer only if you can be certain, otherwise say that you cannot find a supported answer."
        )))

    messages.append(HumanMessage(content=user_message))

    prompt_tokens = count_messages_tokens(messages, settings.openai_model)
    completion_tokens = 0
    total_tokens = prompt_tokens
    tool_calls = []
    answer = ""

    try:
        response = llm.invoke(messages)
        answer = response.content
        llm_output = getattr(response, "llm_output", {}) or {}
        usage = llm_output.get("usage", {}) or {}
        prompt_tokens = usage.get("prompt_tokens", prompt_tokens)
        completion_tokens = usage.get("completion_tokens", count_tokens(answer, settings.openai_model))
        total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)
    except Exception as exc:
        logger.warning("RAGAS topic service LLM call failed: %s", exc)
        answer = (
            f"[LLM call failed: {exc}]\n\n"
            f"Context retrieved:\n{context[:500] if context else 'None'}"
        )
        completion_tokens = 0
        total_tokens = prompt_tokens

    estimated_cost = estimate_cost(total_tokens)
    validation = validate_answer_against_sources(answer, sources)

    return {
        "answer": answer,
        "sources": sources,
        "tool_calls": tool_calls,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost": estimated_cost,
        "validation": validation,
    }


def run_ragas_topic_filtered(topic: str, user_message: str) -> dict[str, Any]:
    user_message = sanitize_user_message(user_message)[:MAX_USER_MESSAGE_CHARS]
    logger.info("RAGAS topic-filtered service: topic=%s query='%s'", topic, user_message[:80])

    context, sources = retrieve_context(f"{topic} {user_message}", top_k=settings.rag_top_k)
    tool_calls = []
    dependency_info = []

    if topic and len(topic) > 3 and topic not in _STOP_WORDS:
        try:
            dependency_rows = query_service_dependencies(topic)
        except Exception as exc:
            logger.warning("RAGAS dependency lookup failed for %s: %s", topic, exc)
            dependency_rows = []

        if dependency_rows:
            dependency_info = dependency_rows
            tool_calls.append(f"query_service_dependencies(service={topic})")

    sandbox_result = None
    if "```python" in user_message:
        start = user_message.find("```python") + 9
        end = user_message.find("```", start)
        if end > start:
            code_block = user_message[start:end].strip()
            sandbox_result = execute_code(code_block)
            tool_calls.append("execute_code(sandbox)")

    messages = [SystemMessage(content=RAGAS_TOPIC_FILTERED_SYSTEM_PROMPT)]
    if context:
        messages.append(SystemMessage(content=f"Topic: {topic}\nRelevant documentation:\n{context}"))
    else:
        messages.append(SystemMessage(content=(
            f"Topic: {topic}\nNo relevant documentation was retrieved. "
            "Answer only if you can be certain, otherwise say that you cannot find a supported answer."
        )))

    if dependency_info:
        dep_text = "\n".join(
            f"- {r['upstream']} → {r['downstream']} ({r['dependency_type']})"
            for r in dependency_info
        )
        messages.append(SystemMessage(content=f"Service dependencies (topic filtered):\n{dep_text}"))

    if sandbox_result:
        messages.append(SystemMessage(content=(
            f"Sandbox execution result:\nStatus: {sandbox_result['status']}\nOutput:\n{sandbox_result['output']}"
        )))

    messages.append(HumanMessage(content=user_message))

    prompt_tokens = count_messages_tokens(messages, settings.openai_model)
    completion_tokens = 0
    total_tokens = prompt_tokens
    answer = ""

    try:
        response = llm.invoke(messages)
        answer = response.content
        llm_output = getattr(response, "llm_output", {}) or {}
        usage = llm_output.get("usage", {}) or {}
        prompt_tokens = usage.get("prompt_tokens", prompt_tokens)
        completion_tokens = usage.get("completion_tokens", count_tokens(answer, settings.openai_model))
        total_tokens = usage.get("total_tokens", prompt_tokens + completion_tokens)
    except Exception as exc:
        logger.warning("RAGAS topic-filtered LLM call failed: %s", exc)
        answer = (
            f"[LLM call failed: {exc}]\n\n"
            f"Context retrieved:\n{context[:500] if context else 'None'}"
        )
        completion_tokens = 0
        total_tokens = prompt_tokens

    estimated_cost = estimate_cost(total_tokens)
    validation = validate_answer_against_sources(answer, sources)

    return {
        "answer": answer,
        "sources": sources,
        "tool_calls": tool_calls,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost": estimated_cost,
        "validation": validation,
    }

print("✅ RAGAS functions defined")

✅ RAGAS functions defined


### 7.1 Demo: Topic-Based RAGAS wrappers

In [25]:
ragas_result = run_ragas_topic_service(
    topic="order-service",
    user_message="Summarize how order-service interacts with payment-service and what the key operational dependencies are.",
)

print("=== RAGAS Topic Service ===")
print(ragas_result["answer"])
print(f"Sources: {ragas_result['sources']}")
print(f"Tool calls: {ragas_result['tool_calls']}")

ragas_filtered_result = run_ragas_topic_filtered(
    topic="payment-service",
    user_message=(
        "Describe how payment-service retry logic should behave and include any relevant dependency information."
    ),
)

print("=== RAGAS Topic-Filtered Service ===")
print(ragas_filtered_result["answer"])
print(f"Sources: {ragas_filtered_result['sources']}")
print(f"Tool calls: {ragas_filtered_result['tool_calls']}")

INFO | RAGAS topic service: topic=order-service query='Summarize how order-service interacts with payment-service and what the key oper'
INFO | Retrying request to /chat/completions in 0.412089 seconds
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | RAGAS topic-filtered service: topic=payment-service query='Describe how payment-service retry logic should behave and include any relevant '


=== RAGAS Topic Service ===
The order-service interacts with the payment-service primarily to process payment transactions. The payment-service uses Stripe to handle these transactions. Key operational dependencies for the payment-service include:

1. **Stripe API**: The payment-service relies on a valid and active Stripe API key to process payments. Any issues with the API key, such as it being invalid or rate-limited, can disrupt payment processing.

2. **Connection Pool**: Proper configuration of the connection pool is crucial. Issues such as `PoolExhausted` exceptions can arise if the connection pool settings in `config/pool.yaml` are not correctly configured.

3. **CloudWatch Logs**: Monitoring CloudWatch logs is essential for identifying and diagnosing issues like 5xx errors.

4. **Database Connection Pool Utilization**: High latency issues might be linked to the database connection pool utilization, which needs to be monitored and managed effectively.

5. **Stripe Endpoint Healt

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


=== RAGAS Topic-Filtered Service ===
The `payment-service` is responsible for processing payment transactions via Stripe. When implementing retry logic for this service, it's crucial to consider both the service's internal operations and its dependencies.

### Retry Logic for Payment-Service

1. **Idempotency**: Ensure that retrying a payment transaction does not result in duplicate charges. Use Stripe's idempotency keys to safely retry requests without causing unintended side effects.

2. **Backoff Strategy**: Implement an exponential backoff strategy for retries. This helps to avoid overwhelming the Stripe API and reduces the risk of hitting rate limits.

3. **Error Handling**: 
   - **5xx Errors**: These indicate server-side issues, either with the payment-service itself or with Stripe. Retry these errors with backoff.
   - **Network Errors**: Transient network issues should be retried, as they may resolve themselves.
   - **4xx Errors**: Typically indicate client-side issues (e.g.,

## 7.2 LangGraph Orchestration — Planner + ReAct Loop

This section implements a lightweight agent orchestration workflow using the `langgraph` package for routing and planning.
The user query is first routed by a planner agent to either the SRE agent or the Engineering agent. The selected agent then uses ReAct-style reasoning and tool calls to answer the question. Follow-up queries are supported in a loop with a maximum of 4 iterations.


In [26]:
import sys
import subprocess
from typing import Any, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage

try:
    import langgraph
    from langgraph.graph import START, StateGraph
    LANGGRAPH_AVAILABLE = True
    print("✅ LangGraph imported:", getattr(langgraph, "__version__", "unknown"))
except ImportError:
    langgraph = None
    START = None
    StateGraph = None
    LANGGRAPH_AVAILABLE = False
    print("⚠️ LangGraph is not installed in this environment.")
    print("Install it with: pip install langgraph langchain langchain-openai")

PLANNER_SYSTEM_PROMPT = """
You are a routing planner for the Agentic RAG platform.
Decide whether each incoming user query should be handled by the SRE agent or the Engineering agent.
Return exactly one of: SRE or Engineering.
If the query is about reliability, incidents, alerts, outages, or operational troubleshooting, choose SRE.
If the query is about architecture, code review, design, implementation, or engineering best practices, choose Engineering.
"""

REACTION_INSTRUCTIONS = """
You are using a ReAct reasoning framework.
Think step-by-step before answering.
If you need to use a tool, name the tool and the action explicitly.
When you provide the final answer, clearly note which tools were used and why.
"""


class OrchestrationState(TypedDict, total=False):
    session_id: str
    current_query: str
    decision: str
    agent: str
    answer: str
    sources: list[str]
    tool_calls: list[str]
    iteration: int
    completed: bool


def parse_planner_decision(text: str) -> str:
    normalized = text.strip().lower()
    if "engineering" in normalized and "sre" not in normalized:
        return "engineering"
    if "sre" in normalized and "engineering" not in normalized:
        return "sre"
    if "engineering" in normalized:
        return "engineering"
    if "sre" in normalized:
        return "sre"
    if any(term in normalized for term in ["incident", "outage", "latency", "error", "failure", "alert"]):
        return "sre"
    return "sre"


def plan_agent_route(user_message: str) -> str:
    """Use the planner agent to choose SRE or Engineering."""
    messages = [
        SystemMessage(content=PLANNER_SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ]

    try:
        response = llm.invoke(messages)
        decision = response.content.strip()
    except Exception as exc:
        logger.warning("Planner LLM call failed: %s", exc)
        decision = "SRE"

    route = parse_planner_decision(decision)
    logger.info("Planner decision: %s -> %s", decision, route)
    return route


def planner_node(state: OrchestrationState) -> dict[str, str]:
    decision = plan_agent_route(state["current_query"])
    return {"decision": decision, "agent": decision}


def agent_execution_node(state: OrchestrationState) -> dict[str, Any]:
    agent = state.get("agent", "sre")
    query = state["current_query"]
    if agent == "engineering":
        result = run_engineering_agent(state["session_id"], query, react=True)
    else:
        result = run_sre_agent(state["session_id"], query, react=True)

    return {
        "answer": result["answer"],
        "sources": result["sources"],
        "tool_calls": result["tool_calls"],
    }


def build_orchestration_graph() -> Any:
    if not LANGGRAPH_AVAILABLE or START is None or StateGraph is None:
        return None

    graph = StateGraph(OrchestrationState)
    graph.add_node("planner", planner_node)
    graph.add_node("agent_executor", agent_execution_node)
    graph.add_edge(START, "planner")
    graph.add_edge("planner", "agent_executor")
    return graph.compile()


def run_langgraph_orchestration(session_id: str, user_message: str, max_iterations: int = 4) -> dict[str, Any]:
    """Run planner-driven orchestration with a LangGraph planner and ReAct agent execution."""
    current_query = sanitize_user_message(user_message)[:MAX_USER_MESSAGE_CHARS]
    interactions: list[dict[str, Any]] = []
    iteration = 0
    last_agent = "unknown"
    compiled_graph = build_orchestration_graph() if LANGGRAPH_AVAILABLE else None

    if LANGGRAPH_AVAILABLE:
        print("✅ LangGraph is available for orchestration.")
    else:
        print("⚠️ LangGraph is not installed; using notebook planner fallback.")

    while iteration < max_iterations and current_query:
        iteration += 1
        state: OrchestrationState = {
            "session_id": session_id,
            "current_query": current_query,
            "iteration": iteration,
        }

        if compiled_graph is not None:
            output = compiled_graph.invoke(state)
            route = output.get("agent", "sre")
            last_agent = route
            result = {
                "answer": output.get("answer", ""),
                "sources": output.get("sources", []),
                "tool_calls": output.get("tool_calls", []),
            }
        else:
            route = plan_agent_route(current_query)
            last_agent = route
            if route == "engineering":
                result = run_engineering_agent(session_id, current_query, react=True)
            else:
                result = run_sre_agent(session_id, current_query, react=True)

        print(f"\n--- Iteration {iteration} / {max_iterations} — routing to {route.upper()} agent ---")
        print("Agent answer:\n", result["answer"])
        print("Sources:", result["sources"])
        print("Tool calls:", result["tool_calls"])

        interactions.append({
            "iteration": iteration,
            "agent": route,
            "query": current_query,
            "answer": result["answer"],
            "sources": result["sources"],
            "tool_calls": result["tool_calls"],
        })

        if iteration >= max_iterations:
            print("\nReached the maximum orchestration iteration limit of 4. Ending flow.")
            break

        follow_up = input("\nDo you want to ask a follow-up question? (yes/no): ").strip().lower()
        if follow_up not in {"yes", "y"}:
            print("Ending orchestration after user choice.")
            break

        next_query = input("Enter your follow-up question: ").strip()
        if not next_query:
            print("No follow-up question entered — ending orchestration.")
            break

        current_query = next_query

    return {
        "session_id": session_id,
        "iterations": iteration,
        "final_agent": last_agent,
        "interactions": interactions,
        "completed": iteration <= max_iterations,
    }


✅ LangGraph imported: unknown


In [27]:
orchestration_result = run_langgraph_orchestration(
    session_id="demo-orch-001",
    user_message=(
        "We are seeing a rising error rate on payment-service and need both on-call guidance and a review of the retry logic. "
        "Please decide whether SRE or Engineering should handle the first response."
    ),
)

print("\n=== Orchestration Summary ===")
print(f"Session ID: {orchestration_result['session_id']}")
print(f"Iterations: {orchestration_result['iterations']}")
print(f"Final agent: {orchestration_result['final_agent']}\n")
for item in orchestration_result["interactions"]:
    print(f"Iteration {item['iteration']} -> {item['agent'].upper()} agent")
    print(f"Query: {item['query']}")
    print(f"Tool calls: {item['tool_calls']}")
    print(f"Answer preview: {item['answer'][:320]!s}\n")


✅ LangGraph is available for orchestration.


INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Planner decision: SRE -> sre
INFO | SRE agent: session=demo-orch-001 query='We are seeing a rising error rate on payment-service and need both on-call guida'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



--- Iteration 1 / 4 — routing to SRE agent ---
Agent answer:
 To address the rising error rate on the payment-service, we should follow these steps:

1. **Immediate Triage by SRE:**
   - **Check for Common Issues:** Refer to the Payment Service Runbook for common issues related to 5xx errors. Specifically, check the connection pool settings in `config/pool.yaml`, verify the Stripe API key's validity and rate limits, and inspect CloudWatch logs for `PoolExhausted` exceptions. [source: runbook_payment_service.md]
   - **Monitor Stripe Endpoint Health:** Verify the health of the downstream Stripe endpoint at https://status.stripe.com to rule out external service issues. [source: runbook_payment_service.md]

2. **Escalation Protocol:**
   - If the issue is not resolved within 15 minutes, escalate to the payments-oncall rotation as per the runbook guidelines. [source: runbook_payment_service.md]

3. **Review Retry Logic:**
   - **Engineering Involvement:** The review of the retry logic sho

### 7a. Test the Engineering Agent

In [28]:
eng_result = run_engineering_agent(
    session_id="demo-eng-001",
    user_message="What are the dependencies of order-service and how does it connect to payment-service?",
)

print("=" * 60)
print("ENGINEERING AGENT RESPONSE")
print("=" * 60)
print(eng_result["answer"])
print(f"\nSources: {eng_result['sources']}")
print(f"Tool calls: {eng_result['tool_calls']}")
print(f"Prompt tokens: {eng_result['prompt_tokens']}")
print(f"Completion tokens: {eng_result['completion_tokens']}")
print(f"Total tokens: {eng_result['total_tokens']}")
print(f"Estimated cost: ${eng_result['estimated_cost']}")

INFO | Engineering agent: session=demo-eng-001 query='What are the dependencies of order-service and how does it connect to payment-se'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


ENGINEERING AGENT RESPONSE
The `order-service` has the following dependencies:

1. **Payment Service**: The `order-service` connects to the `payment-service` via HTTP. This connection is used to process payment transactions, typically involving operations like charging a customer or refunding a payment.

2. **Inventory Service**: The `order-service` connects to the `inventory-service` using gRPC. This connection is likely used to manage inventory levels, check stock availability, and update inventory records as orders are processed.

Additionally, the `notification-service` interacts with the `order-service` asynchronously using Amazon SQS (Simple Queue Service) for sending notifications related to order events.

For the connection to the `payment-service`, ensure that the HTTP endpoints are correctly configured and that the necessary authentication (such as API keys or tokens) is in place to securely communicate with the `payment-service`. If you encounter issues, refer to the [Paymen

In [29]:
eng_result_2 = run_engineering_agent(
    session_id="demo-eng-002",
    user_message=(
        "Review the retry logic below for our payment-service client and suggest improvements:\n\n"
        "```python\nimport time\n\ndef call_payment(payload, retries=3):\n"
        "    for i in range(retries):\n"
        "        try:\n"
        "            resp = http_client.post('/pay', json=payload)\n"
        "            resp.raise_for_status()\n"
        "            return resp.json()\n"
        "        except Exception:\n"
        "            time.sleep(1)\n"
        "    raise RuntimeError('payment call failed after retries')\n\n"
        "print('Retry logic defined')\n```"
    ),
)

print("=" * 60)
print("ENGINEERING AGENT RESPONSE — Code Review + Sandbox")
print("=" * 60)
print(eng_result_2["answer"])
print(f"\nSources: {eng_result_2['sources']}")
print(f"Tool calls: {eng_result_2['tool_calls']}")
print(f"Prompt tokens: {eng_result_2['prompt_tokens']}")
print(f"Completion tokens: {eng_result_2['completion_tokens']}")
print(f"Total tokens: {eng_result_2['total_tokens']}")
print(f"Estimated cost: ${eng_result_2['estimated_cost']}")

INFO | Engineering agent: session=demo-eng-002 query='Review the retry logic below for our payment-service client and suggest improvem'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


ENGINEERING AGENT RESPONSE — Code Review + Sandbox
The current retry logic in your `call_payment` function has a few issues and areas for improvement:

1. **Syntax Error**: The code snippet provided has a syntax error due to improper indentation and placement of the `import` statement.

2. **Exception Handling**: The `except Exception` block is too broad. It's better to catch specific exceptions that you expect might occur, such as network-related errors.

3. **Retry Logic**: The `raise RuntimeError` is inside the loop, which means it will raise an error on the first failure instead of after exhausting all retries.

4. **Exponential Backoff**: Implementing an exponential backoff strategy can be more effective than a fixed sleep time, as it reduces the load on the service during high traffic periods.

5. **Logging**: Adding logging can help in debugging and monitoring retry attempts.

Here's an improved version of the retry logic:

```python
import time
import logging
import requests

d

In [31]:
result_cells = [
    ("SRE agent", sre_result),
    ("Engineering agent", eng_result),
    ("Engineering agent — code review", eng_result_2),
]

for name, result in result_cells:
    print("=" * 80)
    print(f"{name.upper()} MONITORING")
    print("=" * 80)
    print(f"Answer (truncated): {result['answer'][:500]!s}")
    print(f"Sources: {result['sources']}")
    print(f"Tool calls: {result['tool_calls']}")
    print(f"Prompt tokens: {result['prompt_tokens']}")
    print(f"Completion tokens: {result['completion_tokens']}")
    print(f"Total tokens: {result['total_tokens']}")
    print(f"Estimated cost: ${result['estimated_cost']}")
    print(f"Validation: {json.dumps(result['validation'], indent=2)}")
    print()


SRE AGENT MONITORING
Answer (truncated): To address 5xx errors on the payment-service, follow these steps:

1. **Check Connection Pool Settings**: Review the configuration in `config/pool.yaml` to ensure the connection pool settings are correct.

2. **Verify Stripe API Key**: Ensure that the Stripe API key is valid and not rate-limited. This can often be a cause of 5xx errors if the key is invalid or has exceeded its usage limits.

3. **Inspect CloudWatch Logs**: Look for `PoolExhausted` exceptions in the CloudWatch logs. This can indi
Sources: ['runbook_payment_service.md', 'technical-runbook.md', 'company-policy.md', 'product-faq.md']
Tool calls: []
Prompt tokens: 988
Completion tokens: 168
Total tokens: 1156
Estimated cost: $0.00289
Validation: {
  "status": "supported",
  "issues": []
}

ENGINEERING AGENT MONITORING
Answer (truncated): The `order-service` has the following dependencies:

1. **Payment Service**: The `order-service` connects to the `payment-service` via HTTP. This co

In [ ]:
# Explicit fallback/warning behavior when validation is unverified
for name, result in result_cells:
    status = result["validation"]["status"]
    if status in {"unverified", "partially_supported", "unknown"}:
        print("=" * 80)
        print(f"{name.upper()} VALIDATION WARNING")
        print("=" * 80)
        print(f"Status: {status}")
        if result["validation"]["issues"]:
            print("Issues:")
            for issue in result["validation"]["issues"]:
                print(f" - {issue}")
        else:
            print("Issues: no explicit validation issues were detected.")

        if status == "unverified":
            print("⚠️  The answer could not be confidently grounded in retrieved sources.")
            print("Fallback: return a safe response, ask for clarification, or provide a clearly qualified answer.")
        elif status == "partially_supported":
            print("⚠️  The answer is only partially supported by the retrieved context.")
            print("Fallback: verify the unsupported claims before acting on them.")
        else:
            print("⚠️  Validation status is unknown; treat the answer as untrusted until confirmed.")
        print()


ENGINEERING AGENT — CODE REVIEW VALIDATION WARNING
Status: partially_supported
Issues:
 - Answer does not explicitly reference any retrieved source.
⚠️  The answer is only partially supported by the retrieved context.
Fallback: verify the unsupported claims before acting on them.



In [33]:
orchestration_result = run_langgraph_orchestration(
    session_id="demo-orch-002",
    user_message=(
        "What are the dependencies of order-service and how does it connect to payment-service?"
    ),
)

print("\n=== Orchestration Summary ===")
print(f"Session ID: {orchestration_result['session_id']}")
print(f"Iterations: {orchestration_result['iterations']}")
print(f"Final agent: {orchestration_result['final_agent']}\n")
for item in orchestration_result["interactions"]:
    print(f"Iteration {item['iteration']} -> {item['agent'].upper()} agent")
    print(f"Query: {item['query']}")
    print(f"Tool calls: {item['tool_calls']}")
    print(f"Answer preview: {item['answer'][:320]!s}\n")


✅ LangGraph is available for orchestration.


INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Planner decision: Engineering -> engineering
INFO | Engineering agent: session=demo-orch-002 query='What are the dependencies of order-service and how does it connect to payment-se'
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



--- Iteration 1 / 4 — routing to ENGINEERING agent ---
Agent answer:
 The `order-service` has the following dependencies:

1. **Payment Service**: The `order-service` connects to the `payment-service` via HTTP. This connection is crucial for processing payment transactions as part of the order fulfillment process.

2. **Inventory Service**: The `order-service` connects to the `inventory-service` using gRPC. This is used to check and update inventory levels when processing orders.

Additionally, the `order-service` is connected to by the `api-gateway` via HTTP, which serves as the entry point for external requests.

For the connection to the `payment-service`, ensure that the HTTP endpoints are correctly configured and that any necessary authentication, such as API keys or tokens, is properly managed. According to the [Payment Service Runbook](runbook_payment_service.md), you should also monitor for common issues such as 5xx errors and high latency, which might involve checking connect

In [34]:
orchestration_result = run_langgraph_orchestration(
    session_id="demo-orch-003",
    user_message=(
        "Review the retry logic below for our payment-service client and suggest improvements:\n\n"
        "```python\nimport time\n\ndef call_payment(payload, retries=3):\n"
        "    for i in range(retries):\n"
        "        try:\n"
        "            resp = http_client.post('/pay', json=payload)\n"
        "            resp.raise_for_status()\n"
        "            return resp.json()\n"
        "        except Exception:\n"
        "            time.sleep(1)\n"
        "    raise RuntimeError('payment call failed after retries')\n\n"
        "print('Retry logic defined')\n```"
    ),
)

print("\n=== Orchestration Summary ===")
print(f"Session ID: {orchestration_result['session_id']}")
print(f"Iterations: {orchestration_result['iterations']}")
print(f"Final agent: {orchestration_result['final_agent']}\n")
for item in orchestration_result["interactions"]:
    print(f"Iteration {item['iteration']} -> {item['agent'].upper()} agent")
    print(f"Query: {item['query']}")
    print(f"Tool calls: {item['tool_calls']}")
    print(f"Answer preview: {item['answer'][:320]!s}\n")


✅ LangGraph is available for orchestration.


INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Planner decision: Engineering -> engineering
INFO | Engineering agent: session=demo-orch-003 query='Review the retry logic below for our payment-service client and suggest improvem'
INFO | Retrying request to /chat/completions in 0.409034 seconds
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



--- Iteration 1 / 4 — routing to ENGINEERING agent ---
Agent answer:
 The provided retry logic has a few issues and areas for improvement. Let's address them step-by-step:

1. **Syntax Error**: The code snippet has a syntax error due to incorrect placement of the `import` statement and the function definition. Ensure that the `import` statement is at the top of the file.

2. **Exception Handling**: The `except Exception` block currently raises an error immediately after the first failure. This should be adjusted to only raise an error after all retries have been exhausted.

3. **Backoff Strategy**: Implementing an exponential backoff strategy can help reduce the load on the server and improve the chances of success in subsequent retries.

4. **Logging**: Adding logging can help in diagnosing issues when retries occur.

5. **HTTP Client**: Ensure that `http_client` is properly defined and configured before using it.

Here's an improved version of the retry logic:

```python
import time

## 8. Health Check

Mirrors `app/api/routes/health.py` — verifies connectivity to all backend services.

In [51]:
def check_health() -> dict[str, Any]:
    """
    Local health check — mirrors app/api/routes/health.py.
    Checks local Redis, SQLite, LLM client, and ChromaDB.
    """
    services = {}
    overall = "ok"

    # Redis
    try:
        redis_client.ping()
        services["redis"] = "ok (local in-memory)"
    except Exception as exc:
        services["redis"] = f"error: {exc}"
        overall = "degraded"

    # PostgreSQL (SQLite)
    try:
        with _get_db() as conn:
            conn.execute("SELECT 1").fetchone()
        services["postgres"] = "ok (local SQLite)"
    except Exception as exc:
        services["postgres"] = f"error: {exc}"
        overall = "degraded"

    # LLM via LangChain — verify provider selection with a lightweight call
    try:
        llm.invoke([HumanMessage(content="ping")])
        services["llm"] = f"ok (provider={settings.llm_provider}, model={llm_model})"
    except Exception as exc:
        services["llm"] = f"degraded: {exc}"
        overall = "degraded"

    # ChromaDB
    try:
        count = kb.count
        services["chromadb"] = f"ok ({count} chunks indexed)"
    except Exception as exc:
        services["chromadb"] = f"error: {exc}"
        overall = "degraded"

    return {"status": overall, "services": services}


health = check_health()
print("Health Check:")
print(json.dumps(health, indent=2))

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Health Check:
{
  "status": "ok",
  "services": {
    "redis": "ok (local in-memory)",
    "postgres": "ok (local SQLite)",
    "llm": "ok (provider=openai, model=gpt-4o)",
    "chromadb": "ok (1 chunks indexed)"
  }
}


## 9. Streamlit Chat UI

Instead of FastAPI routes (`app/api/routes/chat.py`, `ingest.py`, `documents.py`, `health.py`), we provide a **Streamlit** app that exposes the same functionality through a browser UI.

The cell below writes a `streamlit_app.py` file to disk and provides instructions to run it.

### Features
- **Agent selector** — choose between SRE and Engineering agents
- **Chat interface** — multi-turn conversation with history
- **Document ingestion** — upload `.md` / `.txt` files into the local search index
- **Health dashboard** — shows status of all local services

## 12. Vector Store (Reference)

This notebook uses a local ChromaDB vector store for semantic retrieval.
The following notes describe the local ChromaDB setup used in the demo.

In [52]:
# ── Vector store reference ─────────────────────────────────────────────────
#
# Local ChromaDB vector store:
#     - PersistentClient stores data at data/chroma_rag/
#     - DefaultEmbeddingFunction is used for embeddings
#     - collection.query(query_texts=[...]) performs similarity search
#     - Documents are chunked and embedded locally for retrieval

print("ℹ️  Vector store section is reference-only.")
print(f"   Local demo uses ChromaDB with {kb.count} chunks indexed.")

ℹ️  Vector store section is reference-only.
   Local demo uses ChromaDB with 1 chunks indexed.


## 13. Audit Log — Review Agent Interactions

All agent interactions are persisted in the local SQLite database. This is useful for debugging and reviewing agent behaviour.

In [53]:
with _get_db() as conn:
    rows = conn.execute(
        "SELECT session_id, agent, query, created_at FROM agent_interactions ORDER BY created_at DESC LIMIT 10"
    ).fetchall()

print(f"Recent agent interactions ({len(rows)}):")
print("-" * 90)
for r in rows:
    row = dict(r)
    print(f"  [{row['created_at']}] {row['agent']:12s} session={row['session_id']}")
    print(f"    Q: {row['query'][:80]}")
    print()

Recent agent interactions (5):
------------------------------------------------------------------------------------------
  [2026-05-25 10:36:04] engineering  session=demo-eng-002
    Q: Review the retry logic below for our payment-service client and suggest improvem

  [2026-05-25 10:35:52] engineering  session=demo-eng-001
    Q: What are the dependencies of order-service and how does it connect to payment-se

  [2026-05-25 10:34:38] sre          session=demo-sre-003
    Q: Our monitoring shows Redis connection pool exhaustion on notification-service. C

  [2026-05-25 10:34:35] sre          session=demo-sre-002
    Q: We're observing p99 latency spikes on order-service above 2s. How should I triag

  [2026-05-25 10:34:27] sre          session=demo-sre-001
    Q: We're seeing 5xx errors on payment-service. What should I check first?



---

## Summary

This notebook demonstrates a local, open-source version of the Agentic RAG platform.

| Section | Local implementation |
|---|---|
| Settings | Simplified Pydantic settings from `.env` |
| Schemas | Pydantic models for agent requests and responses |
| LLM Client | `LangChain ChatOpenAI` with standard OpenAI API key |
| Search / Vector Store | ChromaDB persistent vector store with local embeddings |
| Postgres Tool | Local SQLite incident history and dependencies |
| Redis Tool | In-memory dict for conversation history |
| Sandbox Tool | Restricted Python execution in notebook |
| RAG Service | Hybrid local retrieval with keyword search + ChromaDB |
| SRE Agent | Local SRE assistant using local tools and LLM calls |
| Engineering Agent | Local engineering assistant with local tool support |
| API / UI | Streamlit browser UI for chat, ingestion, and health |
| Health Check | Local service checks for Redis, SQLite, LLM, and ChromaDB |

### Next steps
- Add `.md` / `.txt` runbooks to `data/raw-docs/` to enrich the knowledge base
- Set `OPENAI_API_KEY` in `.env` for real LLM responses
- Run `streamlit run notebooks/streamlit_app_noAzure.py` for the interactive chat UI